# Qwen3-8B smoke test

This notebook loads the 4-bit AWQ build of Qwen3-8B with vLLM on a T4 GPU, sends one prompt, and prints tokens per second.

Run it on Kaggle with the accelerator set to GPU T4 and internet enabled. Thinking mode is off, matching the model client.

In [ ]:
import subprocess

gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True)
print(gpu.strip())
if "T4" not in gpu:
    raise RuntimeError("This smoke test expects a T4 GPU. Set the notebook accelerator to GPU T4.")

In [ ]:
import subprocess

subprocess.check_call(["pip", "install", "-q", "vllm>=0.8.5"])

In [ ]:
import time

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

MODEL = "Qwen/Qwen3-8B-AWQ"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
llm = LLM(
    model=MODEL,
    quantization="awq",
    dtype="float16",
    max_model_len=2048,
    gpu_memory_utilization=0.90,
    enforce_eager=True,
)

prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Reply with exactly: hello"}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
params = SamplingParams(temperature=0, max_tokens=32)

started = time.perf_counter()
outputs = llm.generate([prompt], params)
elapsed = time.perf_counter() - started

completion = outputs[0].outputs[0]
completion_tokens = len(completion.token_ids)
print(f"model: qwen")
print(f"text: {completion.text.strip()}")
print(f"prompt_tokens: {len(outputs[0].prompt_token_ids)}")
print(f"completion_tokens: {completion_tokens}")
print(f"tokens_per_sec: {completion_tokens / elapsed:.2f}")